# Notebook: TRIX

**Version**: v1.0-beta 

**Last update**: 2025-01-31

**Authors**: OGS

# TROPHIC STATE INDEX - TRIX

The trophic state depends on the availability of nitrogen and phosphorus for primary production, which in terms determines the phytoplankton biomass and oxygen saturation. In TRIX the nutrients are represented ideally by total nitrogen and total phosphorus; chlorophyll-a is a substitute parameter for phytoplankton biomass, as production is not routinely measured; and the deviation of oxygen saturation from 100% (aD%O) in the productive layer indicates the production intensity of the system. This encompasses both phases of active photosynthesis and phases of prevailing respiration.(European Commission. Joint Research Centre., 2016).


The following notebook should usue the DIVAnd outputs from the eutrophication workbench. 

* disclimer: the notebook is in beta version since the datasets from the workbenches are not yet ready

load the modules, you will probably need to install the following libraries:

In [1]:
# %pip install cmocean
# !pip install plotly
# !pip install dash

In [2]:
import os
import plotly.express as px 
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import dash
from dash import Dash, html, dcc, Input, Output, callback, dash_table
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.gridliner as gridliner
import matplotlib.ticker as mticker
import cartopy.mpl.ticker as cartopyticker
import cmocean
import xarray as xr 
import netCDF4 as nc
from collections import OrderedDict
import ipywidgets as widgets
from IPython.display import display, HTML


## 1. Read DIVAnd Analysis outputs:

We run a test with DIVAnd in the Adriatic sea for two 10 years periods: 2003-2012 and 2013-2022 of in-situ data from emodnet chemistry data collections.

- **Import Statements**:
  - `import xarray as xr`: Imports the xarray library, which is used for working with multi-dimensional arrays and datasets.
  - `import netCDF4 as nc`: Imports the netCDF4 library, which is used as an engine for reading and writing NetCDF files. NetCDF (Network Common Data Form) is a format for array-oriented scientific data.

- **Opening NetCDF Datasets**:
  - The `xr.open_dataset()` function is used to open NetCDF files and create xarray datasets from them.
  - `decode_cf=True` parameter: Ensures that the dataset is interpreted according to CF (Climate and Forecast) metadata conventions, which include handling time coordinates, scale/offset attributes, and more.-

In [ ]:
from pathlib import Path
import xarray as xr
import ipywidgets as widgets
from IPython.display import display, clear_output

base_dir = Path(r"C:\Users\nreyessuarez\OneDrive - OGS\Documenti\GitHub\Blue-Cloud\MEI_VLab\DIVA_climatologies")
assert base_dir.exists(), f"{base_dir} does not exist"

folder_dropdown = widgets.Dropdown(
    options=[(d.name, d) for d in sorted(base_dir.iterdir()) if d.is_dir()],
    description="Select folder:",
    layout=widgets.Layout(width="600px")
)

output = widgets.Output()

def open_climatology_files(folder: Path):
    """
    Finds and opens the four expected files in the chosen folder.
    """
    patterns = {
        "DIN": "*dissolved_inorganic_nitrogen*/*.nc" if False else "*dissolved_inorganic_nitrogen*.nc",
        "CHL": "*chlorophyll-a*.nc",
        "TP":  "*total_phosphorus*.nc",
        "DO":  "*calculatedDOsaturation*.nc"
    }
    datasets = {}
    for key, pattern in patterns.items():
        candidates = sorted(folder.glob(pattern))
        if len(candidates) == 0:
            print(f"[WARN] {key} file not found by pattern: {pattern}")
            datasets[key] = None
            continue
        path = candidates[0]
        datasets[key] = xr.open_dataset(path, decode_cf=True, chunks="auto")
        print(f"[OK] {key} opened: {path.name}")
    return datasets

def on_folder_change(change):
    if change["name"] == "value" and change["new"]:
        with output:
            clear_output()
            selected_folder = Path(change["new"])
            print("Selected folder:", selected_folder)
            ds = open_climatology_files(selected_folder)
            # set globals for same names used in your workflow
            globals().update({
                "ds_DIN": ds["DIN"],
                "ds_CHL": ds["CHL"],
                "ds_TP": ds["TP"],
                "ds_DO": ds["DO"]
            })
            print("Datasets are now assigned to: ds_DIN, ds_CHL, ds_TP, ds_DO")
            print("Preview ds_DO:")
            if ds["DO"] is not None:
                print(ds["DO"])
            else:
                print("ds_DO is not available yet.")

folder_dropdown.observe(on_folder_change, names="value")
display(folder_dropdown, output)

# trigger initial folder listing even if no selection yet
if folder_dropdown.value:
    on_folder_change({"name":"value","new":folder_dropdown.value})


Dropdown(description='Select folder:', layout=Layout(width='600px'), options=(('2003-2012', WindowsPath('C:/Us…

Output()

In [4]:
# ds_DIN = xr.open_dataset(os.path.expanduser('~/blue-cloud-dataspace/MEI/Eutrophication_indicator/new_data/Water_body_dissolved_inorganic_nitrogen_Med_merged03.4Danl.nc'),decode_cf=True)
# ds_CHL = xr.open_dataset(os.path.expanduser('~/blue-cloud-dataspace/MEI/Eutrophication_indicator/new_data/Water_body_chlorophyll-a_Med_merged03.4Danl.nc'),decode_cf=True)
# ds_TP = xr.open_dataset(os.path.expanduser('~/blue-cloud-dataspace/MEI/Eutrophication_indicator/new_data/Water_body_total_phosphorus_Med_merged03.4Danl.nc'),decode_cf=True)
# ds_DO = xr.open_dataset(os.path.expanduser('~/blue-cloud-dataspace/MEI/Eutrophication_indicator/new_data/Water_body_dissolved_oxygen_saturation_Med_merged03.4Danl.nc'),decode_cf=True)

In [18]:
# explore the dataset
ds_DO.info()


xarray.Dataset {
dimensions:
	lon = 348 ;
	lat = 128 ;
	depth = 8 ;
	time = 4 ;
	nv = 2 ;
	observations = 821193 ;

variables:
	float64 lon(lon) ;
		lon:units = degrees_east ;
		lon:standard_name = longitude ;
		lon:long_name = longitude ;
	float64 lat(lat) ;
		lat:units = degrees_north ;
		lat:standard_name = latitude ;
		lat:long_name = latitude ;
	float64 depth(depth) ;
		depth:units = meters ;
		depth:positive = down ;
		depth:standard_name = depth ;
		depth:long_name = depth below sea level ;
	datetime64[ns] time(time) ;
		time:standard_name = time ;
		time:long_name = time ;
		time:climatology = climatology_bounds ;
	datetime64[ns] climatology_bounds(time, nv) ;
	float32 calculatedDOsaturation(time, depth, lat, lon) ;
		calculatedDOsaturation:units = % ;
		calculatedDOsaturation:standard_name = fractional_saturation_of_oxygen_in_sea_water ;
		calculatedDOsaturation:long_name = Water Body Dissolved Oxygen Saturation ;
		calculatedDOsaturation:cell_methods = time: mean within years

### Filtering out outliers:

This process ensures that only relevant data points are considered for each variable, based on predefined criteria for the North Adriatic Sea.

- **General Filtering Process**:
 
  - Values that fall outside the specified range are replaced with `NaN` (Not a Number), effectively filtering them out.

- **Variables and Ranges**:
  - **Chlorophyll_a (`CHLF`)**:
    - Filters values to retain only those between 0 and 4.
  - **Dissolved Inorganic Nitrogen (`DINF`)**:
    - Filters values to retain only those between 0 and 30.
  - **Total Phosphorus (`TPF`)**:
    - Filters values to retain only those between 0 and 1.
  - **Dissolved Oxygen Saturation (`DOF`)**:
    - Filters values to retain only those between 0and 120.

This filtering process is essential for focusing on the range of interest for each variable and discarding outliers or irrelevant data points.

In [19]:
CHLF = ds_CHL['Water body chlorophyll-a_L1']
DINF = ds_DIN['Water body dissolved inorganic nitrogen (DIN)_L1']
TPF = ds_TP['Water body total phosphorus_L1']
DOF = ds_DO['calculatedDOsaturation_L1']

In [7]:
# CHLF = ds_CHL['Water body chlorophyll-a_L1'].where((ds_CHL['Water body chlorophyll-a_L1'] >=0) & (ds_CHL['Water body chlorophyll-a_L1'] <=4))
# DINF = ds_DIN['Water body dissolved inorganic nitrogen (DIN)_L1'].where((ds_DIN['Water body dissolved inorganic nitrogen (DIN)_L1'] >= 0) & (ds_DIN['Water body dissolved inorganic nitrogen (DIN)_L1'] <=30))
# TPF = ds_TP['Water body total phosphorus_L1'].where((ds_TP['Water body total phosphorus_L1'] >=0) & (ds_TP['Water body total phosphorus_L1'] <=1))
# DOF = ds_DO['calculatedDOsaturation_L1'].where((ds_DO['calculatedDOsaturation_L1'] >= 0) & (ds_DO['calculatedDOsaturation_L1'] <=400))

## 2. TRIX calculation

The TRIX index is based on four state variables ($n$), which are directly related to productivity: 

* Chlorophyll-a (Chl, mg $m^{-3}$),
* Oxygen saturation (DO, $\%$),
* Dissolved inorganic nitrogen (DIN, mg m$^{-3}$), and
* Total phosphorus (TP,mg m$^{-3}$)

Total phosphorus (TP) is a measure of all phosphorus found in a sample, whether that phosphorus is dissolved or particulate. Oxygen as the absolute percentage deviation from oxygen saturation (DO, $\%$), dissolved inorganic nitrogen (DIN, mg m$^{-3}$, and total phosphorous (TP, mg m$^{-3}$). 

In particular,

$$
\text{DIN} = \text{N-NO}_{3} + \text{N-NO}_{2} + \text{N-NH}_{4}
$$ 

and 

$$\text{DO} = \left |100-\text{O}_{x} \right |$$
where $\text{O}_{x}$ is the oxygen saturation.

### 2.1 Unit check

In [20]:
print('total phosphorus units:', TPF.units)
print('dissolved inorganic nitrogen units:', DINF.units)
print('chlorophyll-a units:', CHLF.units)
print('dissolved oxygen saturation units:', DOF.units)

total phosphorus units: umol/l
dissolved inorganic nitrogen units: umol/l
chlorophyll-a units: mg/m^3
dissolved oxygen saturation units: %


#### 2.2.1 DIN and TP
from $\mu\text{mol}/\text{l}$ to $\text{mg}/\text{m}^{3}$:
$$\frac{\mu\text{mol}}{l} = \text{atomic weight}_{i}\times\frac{\text{mg}}{\text{m}^{3}}$$
where i = P or N.
- Atomic weight of N =  14.00672
- Atomic weight of P =  30.97376

In [21]:
TP = 30.97376*TPF #convert from umol/l to mg/m3

In [22]:
DIN = 14.00672*DINF #convert from umol/l to mg/m3

#### 2.2.2 Absolute percentage deviation from oxygen saturation (DO, $\%$)
$$\text{DO} = \left |100-\text{O}_{x} \right |$$
where $O_{x}$ is the oxygen saturation.

In [23]:
DO = abs(100 - DOF)

In [24]:
CHL=CHLF

Each state variable is scaled by the highest ($U_{i}$) and the lowest ($L_{i}$ ) values in the data time series, and TRIX is defined as:

$$\mathrm{TRIX } = \frac{k}{n}\sum_{i=1}^{n}\frac{(\log M_{i} - \log L_{i})}{(\log U_{i} - \log L_{i})}$$

where $k=10$ is another scaling factor; $n=4$ is the number of state variables considered; and $M_{i}$ are the observed Chl, DO, DIN, and TP values.

Vollenweider et al. (1998) further simplified the TRIX formula by assuming (on the basis of the data used) that the difference $(\log U_{i} -\log L_{i})$ was equal to 3 for all state variables. Therefore, considering $k=10$, $n=4$, and the specific $\log L_{i}$ values, the TRIX formula was rewritten as follows:

$$\mathrm{TRIX} = \frac{10}{12}\left [ (\log M_\text{Chl} + 0.5) + (\log M_\text{DO} + 1) + (\log M_\text{DIN}-0.5) + (\log M_\text{TP} + 0.5) \right ]$$

Or

$$\mathrm{TRIX} = \frac{1}{1.2}\left [ (\log (M_\text{Chl} M_\text{DO} M_\text{DIN} M_\text{TP}) + 1.5 \right ]$$

The latest equation gives the TRIX index currently used by ARPAE and adopted by the Italian national legislation (D.L 260/2010).

In [25]:
TRIX = abs((np.log10(CHL*DIN*TP*DO)+1.5)/1.2)

In [29]:
# # max and min of TRIX
# print('TRIX max:', TRIX.max().values)
# print('TRIX min:', TRIX.min().values)
TRIX.depth.values

array([  0.,   5.,  10.,  20.,  30.,  50.,  75., 100.])

In [26]:
# import geopandas as gpd
# from pathlib import Path

# eez_path = Path(r"C:\Users\nreyessuarez\OneDrive - OGS\Documenti\GitHub\Blue-Cloud\MEI_VLab\DIVA_climatologies\eez_12nm_v4.shp")
# eez = gpd.read_file(eez_path)

# if eez.crs is None:
#     eez = eez.set_crs("EPSG:4326", allow_override=True)

# eez = eez.to_crs("EPSG:4326")  # now safe

In [27]:

# import geopandas as gpd
# from shapely.geometry import Point

# def trix_slice_to_gdf(t_idx, d_idx):
#     arr = TRIX.isel(time=t_idx, depth=d_idx)
#     lon = arr['lon'].values
#     lat = arr['lat'].values
#     values = arr.values

#     xs, ys = np.meshgrid(lon, lat)
#     df = pd.DataFrame({
#         'lon': xs.ravel(),
#         'lat': ys.ravel(),
#         'trix': values.ravel()
#     })
#     df = df[~np.isnan(df['trix'])]

#     gdf = gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df.lon, df.lat)], crs="EPSG:4326")
#     return gdf


In [32]:
import os

# # # Obtain the JupyterHub service prefix from the environment variable
# prefix = os.environ['JUPYTERHUB_SERVICE_PREFIX']

# # # Construct the URL for accessing the app through JupyterHub
# url = f'https://jupyterhub.d4science.org/{prefix}proxy/8050/'


from dash import _jupyter

# Define colors for the app's background and text
colors = {
    'background': '#111111',
    'text': '#fdae61'
}

# TRIX eutrophication indicator conditions and categories
data = OrderedDict(
    [
        ("Conditions", ["Oligotrophic", "Oligotrophic", "Oligotrophic", "Eutrophic"]),
        ("TRIX units", ["< 4", "4 << 5", "5 << 6", "> 6"]),
        ("Trophic state", ["Elevated", "Good", "Mediocre", "Bad"]),
    ]
)

# Create a DataFrame to display in the app's table
df = pd.DataFrame(
    OrderedDict([(name, col_data) for (name, col_data) in data.items()])
)

# Initialize the Dash app
app = Dash(__name__)




# Layout of the app, with interactive components and a table
app.layout = html.Div([
    html.H1('TRIX Eutrophication Indicator', style={'textAlign': 'center', 'color': colors['text']}),
    html.H3("Select Season and Depth:"),
    dcc.RadioItems(
        id='i',
        options=[
            {'label': 'JFM', 'value': 0},
            {'label': 'AMJ', 'value': 1},
            {'label': 'JAS', 'value': 2},
            {'label': 'OND', 'value': 3},
            
        ],
        value=0,
        inline=True
    ),
    html.Div(
        dcc.Slider(0,12,1,
            
            id='j',
            marks={i: f'Depth {TRIX.depth[i].values} m' for i in range(0, len(TRIX.depth))},
            value=0,
            included=False,
            vertical=False
        ),
        style={'width': '1000px', 'height': '30px', 'flex': 2, 'margin': 20, 'padding': 20, 'alignItems': 'flex-start'}
    ),
    html.Div([
        html.Div(
            dcc.Graph(id="graph"),
            style={'flex': 1, 'padding': '0 20px 0 0'}
        ),
        html.Div(
            dash_table.DataTable(
                data=df.to_dict('records'),
                columns=[{'id': c, 'name': c} for c in df.columns],
                page_action='none',
                style_table={'height': '200px', 'overflowY': 'auto'},
                fixed_rows={'headers': True},
                style_cell={'minWidth': 10, 'maxWidth': 10, 'width': 10},
                
            ),
            style={'width': '400px', 'margin': '100px', 'alignItems': 'flex-start'}
        )
    ], style={'display': 'flex', 'flexDirection': 'row', 'alignItems': 'flex-start'})]) 


           

@app.callback(
    
    Output("graph", "figure"),
    Input("i", "value"), Input("j", "value"),
    
)
def display_choropleth(i, j):
    # Load data
    df = pd.read_csv(r"C:\Users\nreyessuarez\OneDrive - OGS\Documenti\GitHub\Blue-Cloud\MEI_VLab\DIVA_climatologies\12M_med.csv")
    
    

    # Flatten the data for scattermapbox
    z_values = np.ravel(TRIX[i, j, :, :])
    lon_values, lat_values = np.meshgrid(TRIX.lon, TRIX.lat)
    lon_values = lon_values.ravel()
    lat_values = lat_values.ravel()
    
    # Round and prepare coordinate comparison
    df['X'] = df['X'].round(2)
    df['Y'] = df['Y'].round(2)
         
    #Handle NaN values in z_values
    nan_mask = np.isnan(z_values)
    z_values_with_transparency = np.where(nan_mask, -999, z_values)
    
    #Create a DataFrame for the meshgrid values
    meshgrid_df = pd.DataFrame({
        'lon': lon_values,
        'lat': lat_values,
        'z': z_values_with_transparency
    })
    
    # Merge the meshgrid data with df to get corresponding z values
    merged_df = pd.merge(df, meshgrid_df, left_on=['X', 'Y'], right_on=['lon', 'lat'], how='left')
    

    # Handle cases where merged z values might be NaN
    merged_df.fillna({'z':-999}, inplace=True)
    
    # Create a scattermapbox trace for the heatmap
    heatmap_trace = go.Scattermap(
        lon=df['X'],
        lat=df['Y'],
        mode='markers',
        marker=dict(
            size=5,
            color=merged_df['z'],
            colorscale=[
                (0.00, "rgba(0,0,0,0)"),
                (0.14, "#5B2C6F"),
                (0.28, "#3498DB"),
                (0.42, "#1ABC9C"),
                (0.56, "#27AE60"),
                (0.70, "#F1C40F"),
                (0.84, "#E67E22"),
                (1.00, "#E74C3C")
            ],
            showscale=True,
            cmin=4,
            cmax=6,
            colorbar=dict(title="TRIX"),
            opacity=1
        ),
        text=merged_df['z'],
        connectgaps=True
    )

    # Create the figure
    fig = go.Figure(data=[heatmap_trace])

   
    fig.update_layout(
        mapbox=dict(
            style="satellite",
            center={"lon": 15.22, "lat": 43.14},
            zoom=3
        ),
        title_text='TRIX by Season',
        height=1300,
        width=1300,
    )
   
    return fig

# # Update base path for Jupyter server
# _jupyter._jupyter_config.update({"base_subpath": prefix})
# if __name__ == '__main__':
#     app.run(debug=True, jupyter_server_url=url, dev_tools_props_check_interval=50000)